In [7]:
!pip -q install pandas numpy scikit-learn beautifulsoup4 lxml tqdm


In [8]:
import re
import os
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

tqdm.pandas()


In [9]:
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print("Uploaded:", file_name)
df = pd.read_csv(file_name)
df.head()


Saving high_accuracy_emails.csv to high_accuracy_emails (1).csv
Uploaded: high_accuracy_emails (1).csv


,id,subject,body,text,category,category_id
0,promotions_582,Anniversary Special: Buy one get one free,"As our loyal customer, get exclusive $60 off $75+: example.com/6058 Offer code: WELCOME20.","Anniversary Special: Buy one get one free As our loyal customer, get exclusive $60 off $75+: example.com/6058 Offer ...",promotions,1
1,spam_1629,Your Amazon was used on new device,Your $5000 refund is processed. Claim: bit.ly/fakeprize Complete within 48hrshrs.,Your Amazon was used on new device Your $5000 refund is processed. Claim: bit.ly/fakeprize Complete within 48hrshrs.,spam,3
2,spam_322,Re: Your Google inquiry,"Hi, following up about your Google application. Rates dropped to 0.9%! Apply now: phishing-site.com. STOP to opt out.","Re: Your Google inquiry Hi, following up about your Google application. Rates dropped to 0.9%! Apply now: phishing-s...",spam,3
3,social_media_80,Digital Ritual Experience Creation,Cross-cultural ceremony design. Join: virtualrituals.space Anthropological framework.,Digital Ritual Experience Creation Cross-cultural ceremony design. Join: virtualrituals.space Anthropological framew...,social_media,2
4,forum_1351,"Your post was moved to ""Programming Help""","Trending: ""cooking"" (258 comments). View: support.site/ticket/456.","Your post was moved to ""Programming Help"" Trending: ""cooking"" (258 comments). View: support.site/ticket/456.",forum,0


In [10]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.sample(5)


Shape: (10780, 6)
Columns: ['id', 'subject', 'body', 'text', 'category', 'category_id']


,id,subject,body,text,category,category_id
4147,social_media_1554,Watch later: Recommended article,"See which post: social.com/posts/photo123 Comments: ""Great insight!"" Disable notifications: social.com/settings.","Watch later: Recommended article See which post: social.com/posts/photo123 Comments: ""Great insight!"" Disable notifi...",social_media,2
3777,spam_1575,Google: Password expiration,Unusual login detected from China. Secure account: scam-delivery.com NOW or account suspended.,Google: Password expiration Unusual login detected from China. Secure account: scam-delivery.com NOW or account susp...,spam,3
8597,updates_1365,Security notice: Suspicious login blocked,Items cancelled via UPS (tracking: 1Z999AA103456789). Estimated delivery: Sep 1. Track: banking.com/statements.,Security notice: Suspicious login blocked Items cancelled via UPS (tracking: 1Z999AA103456789). Estimated delivery: ...,updates,4
2259,social_media_812,Sarah Johnson liked your video,View profile: social.com/profile/john123 Accept/reject: social.com/friends/pending.,Sarah Johnson liked your video View profile: social.com/profile/john123 Accept/reject: social.com/friends/pending.,social_media,2
1078,spam_1132,Missed delivery confirmation ZX-902,Your package couldn't be delivered. Tracking# AB-123. Update now: bit.ly/fakeprize Do not ignore!,Missed delivery confirmation ZX-902 Your package couldn't be delivered. Tracking# AB-123. Update now: bit.ly/fakepri...,spam,3


In [11]:
df.columns = [c.strip().lower() for c in df.columns]

# Possible column guesses
possible_subject = ["subject", "email_subject", "title"]
possible_body    = ["body", "email_body", "message", "text", "content"]

def pick_col(possible, cols):
    for c in possible:
        if c in cols:
            return c
    return None

subject_col = pick_col(possible_subject, df.columns)
body_col    = pick_col(possible_body, df.columns)

print("Detected subject_col:", subject_col)
print("Detected body_col:", body_col)

if body_col is None:
    raise ValueError("Body/text column not found. Rename your dataset column to one of: " + str(possible_body))

if subject_col is None:
    # If subject doesn't exist, create empty subject
    df["subject"] = ""
    subject_col = "subject"


Detected subject_col: subject
Detected body_col: body


In [12]:
df["subject"] = df[subject_col].fillna("").astype(str)
df["body"]    = df[body_col].fillna("").astype(str)

df["raw_text"] = (df["subject"].str.strip() + " " + df["body"].str.strip()).str.strip()
df[["subject", "body", "raw_text"]].head()


,subject,body,raw_text
0,Anniversary Special: Buy one get one free,"As our loyal customer, get exclusive $60 off $75+: example.com/6058 Offer code: WELCOME20.","Anniversary Special: Buy one get one free As our loyal customer, get exclusive $60 off $75+: example.com/6058 Offer ..."
1,Your Amazon was used on new device,Your $5000 refund is processed. Claim: bit.ly/fakeprize Complete within 48hrshrs.,Your Amazon was used on new device Your $5000 refund is processed. Claim: bit.ly/fakeprize Complete within 48hrshrs.
2,Re: Your Google inquiry,"Hi, following up about your Google application. Rates dropped to 0.9%! Apply now: phishing-site.com. STOP to opt out.","Re: Your Google inquiry Hi, following up about your Google application. Rates dropped to 0.9%! Apply now: phishing-s..."
3,Digital Ritual Experience Creation,Cross-cultural ceremony design. Join: virtualrituals.space Anthropological framework.,Digital Ritual Experience Creation Cross-cultural ceremony design. Join: virtualrituals.space Anthropological framew...
4,"Your post was moved to ""Programming Help""","Trending: ""cooking"" (258 comments). View: support.site/ticket/456.","Your post was moved to ""Programming Help"" Trending: ""cooking"" (258 comments). View: support.site/ticket/456."


In [13]:
URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
PHONE_RE = re.compile(r"\b(\+?\d[\d\-\s]{8,}\d)\b")

SIGNOFF_PATTERNS = [
    r"\bregards\b", r"\bkind regards\b", r"\bthanks\b", r"\bthank you\b",
    r"\bsincerely\b", r"\bbest\b", r"\bbest regards\b", r"\byours truly\b"
]

def strip_html(text: str) -> str:
    # remove html tags safely
    soup = BeautifulSoup(text, "lxml")
    return soup.get_text(" ")

def normalize_text(text: str) -> str:
    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = PHONE_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def remove_forward_reply_blocks(text: str) -> str:
    # remove typical email reply chains
    patterns = [
        r"-----original message-----.*",
        r"from:.*\n.*(sent:|date:).*",
        r"on .* wrote:.*"
    ]
    for p in patterns:
        text = re.sub(p, " ", text, flags=re.IGNORECASE | re.DOTALL)
    return text

def remove_signature(text: str) -> str:
    # cut off after signoff words if appears near end
    for pat in SIGNOFF_PATTERNS:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m and (len(text) - m.start()) < 250:  # signoff near end
            return text[:m.start()].strip()
    return text

def clean_email(text: str) -> str:
    text = strip_html(text)
    text = remove_forward_reply_blocks(text)
    text = normalize_text(text)
    text = remove_signature(text)
    # keep basic characters
    text = re.sub(r"[^a-z0-9\s\.\,\!\?\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [14]:
df["clean_text"] = df["raw_text"].progress_apply(clean_email)

df[["raw_text", "clean_text"]].sample(5)


100%|██████████| 10780/10780 [00:03<00:00, 3272.81it/s]


,raw_text,clean_text
10621,Netflix Refund Notification #R-2873-UK Unusual login detected from China. Secure account: scam-delivery.com NOW or a...,netflix refund notification r-2873-uk unusual login detected from china. secure account scam-delivery.com now or acc...
7739,"[Moderator] Your report was processed Thread: ""Discussion 428"" started by DataWiz. Settings: support.site/ticket/456.",moderator your report was processed thread discussion 428 started by datawiz. settings support.site ticket 456.
446,Software update available (v3.1) Our service will be unavailable during this window. Updates include bug fixes and n...,software update available v3.1 our service will be unavailable during this window. updates include bug fixes and new...
8071,"Verification required: 321516 Use 603613 as your verification code. Device: Samsung Galaxy S22. Location: Chicago, IL.","verification required 321516 use 603613 as your verification code. device samsung galaxy s22. location chicago, il."
4691,Spotify Family Plan Member Removal Alex Johnson requests removal. Confirm: spotify-family.manage/member=2378 Else pl...,spotify family plan member removal alex johnson requests removal. confirm spotify-family.manage member 2378 else pla...


In [15]:
df["clean_len"] = df["clean_text"].str.split().apply(len)
df = df[df["clean_len"] >= 3].copy()

print("After removing short emails:", df.shape)
df[["clean_text", "clean_len"]].head()


After removing short emails: (10778, 9)


,clean_text,clean_len
0,"anniversary special buy one get one free as our loyal customer, get exclusive 60 off 75 example.com 6058 offer code ...",21
1,your amazon was used on new device your 5000 refund is processed. claim bit.ly fakeprize complete within 48hrshrs.,18
2,"re your google inquiry hi, following up about your google application. rates dropped to 0.9 ! apply now phishing-sit...",23
3,digital ritual experience creation cross-cultural ceremony design. join virtualrituals.space anthropological framework.,11
4,your post was moved to programming help trending cooking 258 comments . view support.site ticket 456.,16


In [16]:
label_cols = [c for c in df.columns if c in ["category", "label", "class", "urgency", "priority"]]
print("Possible existing label columns:", label_cols)


Possible existing label columns: ['category']


In [17]:
CATEGORY_KEYWORDS = {
    "complaints": [
        "not working", "issue", "problem", "error", "failed", "broken", "complaint",
        "bad service", "delay", "refund", "charged", "billing issue", "cancel"
    ],
    "requests": [
        "please", "could you", "can you", "request", "need", "want", "help me",
        "support", "assist", "how to", "guide", "update", "change"
    ],
    "feedback": [
        "feedback", "suggestion", "appreciate", "thank", "great", "good", "love",
        "improve", "rating", "review"
    ],
    "spam": [
        "win", "free", "offer", "click", "buy now", "limited time", "promo",
        "congratulations", "prize", "lottery", "urgent response needed"
    ]
}

def label_category(text: str) -> str:
    t = text.lower()
    scores = {}
    for cat, kws in CATEGORY_KEYWORDS.items():
        scores[cat] = sum(1 for kw in kws if kw in t)
    best_cat = max(scores, key=scores.get)
    if scores[best_cat] == 0:
        # default heuristic: if contains promo style => spam else request
        return "requests"
    return best_cat

df["category"] = df["clean_text"].progress_apply(label_category)
df["category"].value_counts()


100%|██████████| 10778/10778 [00:00<00:00, 50965.92it/s]


,count
category,
requests,7794
spam,1520
complaints,1046
feedback,418


In [18]:
HIGH_URGENCY = ["urgent", "asap", "immediately", "not working", "down", "failed", "can't access", "service outage"]
MED_URGENCY  = ["soon", "priority", "quick", "delay", "pending", "follow up"]
LOW_URGENCY  = ["whenever", "no rush", "later", "just asking", "FYI"]

def label_urgency(text: str) -> str:
    t = text.lower()
    if any(k in t for k in HIGH_URGENCY):
        return "high"
    if any(k in t for k in MED_URGENCY):
        return "medium"
    if any(k in t for k in LOW_URGENCY):
        return "low"
    # fallback: complaints often higher
    return "medium"

df["urgency"] = df["clean_text"].progress_apply(label_urgency)
df["urgency"].value_counts()


100%|██████████| 10778/10778 [00:00<00:00, 152750.00it/s]


,count
urgency,
medium,9685
high,1019
low,74


In [19]:
if "date" not in df.columns:
    df["date"] = pd.Timestamp.today().date()

df[["date", "category", "urgency", "clean_text"]].head()


,date,category,urgency,clean_text
0,2026-02-10,spam,medium,"anniversary special buy one get one free as our loyal customer, get exclusive 60 off 75 example.com 6058 offer code ..."
1,2026-02-10,complaints,medium,your amazon was used on new device your 5000 refund is processed. claim bit.ly fakeprize complete within 48hrshrs.
2,2026-02-10,spam,medium,"re your google inquiry hi, following up about your google application. rates dropped to 0.9 ! apply now phishing-sit..."
3,2026-02-10,requests,medium,digital ritual experience creation cross-cultural ceremony design. join virtualrituals.space anthropological framework.
4,2026-02-10,requests,medium,your post was moved to programming help trending cooking 258 comments . view support.site ticket 456.


In [20]:
final_df = df[["subject", "body", "clean_text", "category", "urgency", "date"]].copy()
final_df.drop_duplicates(subset=["clean_text"], inplace=True)

print("Final dataset shape:", final_df.shape)
final_df.sample(5)


Final dataset shape: (9979, 6)


,subject,body,clean_text,category,urgency,date
3166,Two-step verification code: 603613,"Token: 627259 expires at 09:45 EST. For support, contact support@service.com.","two-step verification code 603613 token 627259 expires at 09 45 est. for support, contact .",requests,medium,2026-02-10
1708,Netflix investment opportunity 300% ROI,"iPad Pro logged in 2hrs minutes ago. Verify: phishing-site.com If not you, secure account.","netflix investment opportunity 300 roi ipad pro logged in 2hrs minutes ago. verify phishing-site.com if not you, sec...",requests,medium,2026-02-10
9001,Authentication token: 417380,"Use 426706 as your verification code. Device: Samsung Galaxy S22. Location: Chicago, IL.","authentication token 417380 use 426706 as your verification code. device samsung galaxy s22. location chicago, il.",requests,medium,2026-02-10
5501,Your invoice is available,Unusual login detected from Romania. Secure account: phishing-site.com NOW or account suspended.,your invoice is available unusual login detected from romania. secure account phishing-site.com now or account suspe...,requests,medium,2026-02-10
6879,Temporary PIN: 286745,"For transaction authorization, enter: 321591. Do not share this number.","temporary pin 286745 for transaction authorization, enter 321591. do not share this number.",requests,medium,2026-02-10


In [21]:
out_path = "processed_emails.csv"
final_df.to_csv(out_path, index=False)
print("Saved:", out_path)


Saved: processed_emails.csv


In [22]:
from google.colab import files
files.download("processed_emails.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>